# EDA — `bronze.yf_pull_log`

One row per symbol requested from Yahoo, including every symbol that came back empty.

This is the only table in the warehouse that records the trusts which **no longer
exist**. Without it they would leave no trace anywhere, which is exactly how survivorship
bias gets into a study unnoticed.

## 1. What Yahoo answered

In [0]:
%sql
SELECT asset_class, status, COUNT(*) AS symbols
FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
GROUP BY asset_class, status
ORDER BY asset_class, status;

In [0]:
%sql
-- The companies Yahoo will not serve. This list is the survivorship evidence.
SELECT source_ticker, requested_symbol, message
FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
WHERE asset_class = 'trust' AND status = 'NODATA'
ORDER BY source_ticker;

**122 symbols requested: trusts 100 OK / 18 NODATA, index 3 OK / 1 NODATA.**

Of the 18 trusts, sixteen return Yahoo's own `No data found, symbol may be delisted`.
Two — `APAX` and `HET` — return an empty frame instead, which lands them in the same
bucket for a slightly different reason.

Only two of the eighteen, `BCPT` and `CSH`, can be recovered from the CSV archive. The
other sixteen are simply gone.

## 2. Currency

The price rows do not carry their unit, which is why the pull captured it per symbol.
Silver cannot rescale a number whose unit it cannot see.

In [0]:
%sql
SELECT currency, COUNT(*) AS symbols
FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
WHERE asset_class = 'trust' AND status = 'OK'
GROUP BY currency
ORDER BY symbols DESC;

In [0]:
%sql
-- Name the four odd ones out, so the Silver decision is about known trusts.
SELECT source_ticker, currency, row_count, first_bar, last_bar
FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
WHERE asset_class = 'trust' AND status = 'OK' AND currency <> 'GBp'
ORDER BY currency, source_ticker;

**96 GBp, 3 USD, 1 EUR.**

Since the beat rate compares *percentage returns* rather than price levels, a consistent
unit within a symbol is all that is strictly required — a trust quoted in dollars still
has a valid dollar return. Whether to convert, leave, or exclude the four is **open for
Silver**, and it is a small decision affecting four trusts out of a hundred.

## 3. Horizon denominators

The beat rate is a percentage, so it needs a denominator: how many trusts have enough
history to be judged over 15, 10, 5 and 3 years. These four numbers belong on the record
before Gold is built, because they are what every headline figure is divided by.

In [0]:
%sql
WITH span AS (
  SELECT symbol,
         MONTHS_BETWEEN(MAX(`Date`), MIN(`Date`)) / 12 AS years
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  GROUP BY symbol
)
SELECT SUM(CASE WHEN years >= 15 THEN 1 ELSE 0 END) AS eligible_15y,
       SUM(CASE WHEN years >= 10 THEN 1 ELSE 0 END) AS eligible_10y,
       SUM(CASE WHEN years >= 5  THEN 1 ELSE 0 END) AS eligible_5y,
       SUM(CASE WHEN years >= 3  THEN 1 ELSE 0 END) AS eligible_3y,
       COUNT(*)                                     AS symbols_with_any_prices
FROM span;

Expect roughly **75 / 85 / 95 / 96** against 100 symbols with any prices at all, plus the
2 archived delisted trusts where their own lifespan allows.

Note these count *span* from first bar to last, not whether the history is continuous or
whether the endpoints survive the outlier rule. Silver will produce the final
denominators; these are the ceiling.

## 4. Does the log agree with the price table?

The log says how many rows each symbol returned. The price table should hold exactly that
many. A mismatch would mean rows were lost between the pull and the warehouse.

In [0]:
%sql
WITH logged AS (
  SELECT source_ticker, CAST(row_count AS INT) AS claimed
  FROM `index-vs-trust-pipeline`.bronze.yf_pull_log
  WHERE asset_class = 'trust' AND status = 'OK'
),
actual AS (
  SELECT source_ticker, COUNT(*) AS landed
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  GROUP BY source_ticker
)
SELECT COUNT(*)                                                AS symbols_checked,
       SUM(CASE WHEN l.claimed = a.landed THEN 1 ELSE 0 END)   AS agreeing,
       SUM(CASE WHEN l.claimed <> a.landed THEN 1 ELSE 0 END)  AS disagreeing
FROM logged l
FULL OUTER JOIN actual a ON a.source_ticker = l.source_ticker;

**100 symbols checked, 100 agreeing, 0 disagreeing** is the expected result. This is a
genuine reconciliation: the log was written by the Python pull, the price rows by a
separate Spark write, and they were never compared until now.

---

## Findings

| # | Finding | Status |
|---|---|---|
| 5.1 | 122 symbols requested. Trusts 100 OK / 18 NODATA; index 3 OK / 1 NODATA (SPLG). | **settled** |
| 5.2 | 16 of the 18 give Yahoo's delisted message; `APAX` and `HET` return an empty frame. | **settled** |
| 5.3 | Only `BCPT` and `CSH` are recoverable from the archive — the survivorship cohort is 2. | **settled** |
| 5.4 | Currency: 96 GBp, 3 USD, 1 EUR. Named above. | **open** |
| 5.5 | Horizon ceilings: roughly 75 / 85 / 95 / 96 symbols for 15 / 10 / 5 / 3 years. | **settled** |
| 5.6 | The log's row counts reconcile exactly with the price table. | **settled** |

5.4 is the only decision left here, and it affects four trusts.